In [ ]:
import torch
import torch.nn as nn

print(torch.__version__)

2.10.0+cu128


In [ ]:
import torch
import torch.nn as nn

class STGCNBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(STGCNBlock, self).__init__()

        # Spatial (관절 관계)
        self.spatial = nn.Conv2d(in_channels, out_channels, kernel_size=(1,1))

        # Temporal (시간 방향)
        self.temporal = nn.Conv2d(out_channels, out_channels, kernel_size=(9,1), padding=(4,0))

        self.bn = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU()

        # Residual
        if in_channels != out_channels:
            self.residual = nn.Conv2d(in_channels, out_channels, kernel_size=(1,1))
        else:
            self.residual = nn.Identity()

    def forward(self, x):
        res = self.residual(x)
        x = self.spatial(x)
        x = self.temporal(x)
        x = self.bn(x)
        x = self.relu(x)
        return x + res

In [ ]:
class STGCN(nn.Module):
    def __init__(self, num_class=10):
        super(STGCN, self).__init__()

        self.data_bn = nn.BatchNorm2d(3)

        self.layer1 = STGCNBlock(3, 64)
        self.layer2 = STGCNBlock(64, 64)
        self.layer3 = STGCNBlock(64, 128)
        self.layer4 = STGCNBlock(128, 256)

        self.global_pool = nn.AdaptiveAvgPool2d((1,1))
        self.fc = nn.Linear(256, num_class)

    def forward(self, x):
        # x shape: (N, C, T, V)

        x = self.data_bn(x)

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)

        x = self.global_pool(x)
        x = x.view(x.size(0), -1)

        x = self.fc(x)

        return x

In [ ]:
dummy = torch.randn(1, 3, 100, 25)  # (N, C, T, V) (배치,좌표,시간,관절)
#1명, 3차원 좌표, 100프레임, 25개 관절

model = STGCN(num_class=5)
output = model(dummy)

print("출력 shape:", output.shape)
# 5개의 클래스 점수

출력 shape: torch.Size([1, 5])
